# Data Mining – Spark Data Lake Assignment
**Dataset:** `sales.csv` | **Environment:** Google Colab + PySpark

Workflow: **Bronze** (raw, unchanged) → **Spark inspection** → **Silver** (basic preprocessing) → **Verification**.

## 1. Install PySpark & create the Data Lake structure

In [1]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("pyspark") is None:      # Colab: installs PySpark; skipped if already present
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])
import pyspark
print("PySpark", pyspark.__version__)

PySpark 3.5.1


In [2]:
import os, shutil, glob

BASE   = "data_lake"
BRONZE = f"{BASE}/bronze"
SILVER = f"{BASE}/silver/preprocessed_sales"
os.makedirs(BRONZE, exist_ok=True)
os.makedirs(SILVER, exist_ok=True)

# --- Get sales.csv (upload in Colab if it is not already in the working directory) ---
if not os.path.exists("sales.csv"):
    try:
        from google.colab import files
        print("Please upload sales.csv ...")
        up = files.upload()
        name = list(up.keys())[0]
        if name != "sales.csv":
            os.rename(name, "sales.csv")
    except ImportError:
        raise FileNotFoundError("Place sales.csv next to this notebook.")

# Bronze = byte-for-byte copy of the original file (never modified)
shutil.copyfile("sales.csv", f"{BRONZE}/sales.csv")

for root, dirs, fs in os.walk(BASE):
    print(root + "/", fs)

data_lake/ []
data_lake/bronze/ ['sales.csv']
data_lake/silver/ []
data_lake/silver/preprocessed_sales/ []


## 2. Load and inspect with Spark

In [3]:
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import *

spark = (SparkSession.builder.appName("SalesDataLake").master("local[*]")
         .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")   # invalid dates -> NULL (no exceptions)
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Picked up JAVA_TOOL_OPTIONS: -XX:+IgnoreUnrecognizedVMOptions -XX:-UseBASE64Intrinsics


Picked up JAVA_TOOL_OPTIONS: -XX:+IgnoreUnrecognizedVMOptions -XX:-UseBASE64Intrinsics


26/11/23 12:52:35 WARN Utils: Your hostname, vm resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/11/23 12:52:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/23 12:52:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.1


In [4]:
# Read RAW data as strings so that dirty values (bad numbers / dates) are NOT silently coerced on load
raw = (spark.read.option("header", True).option("inferSchema", False)
       .option("multiLine", False).csv(f"{BRONZE}/sales.csv"))

raw.printSchema()
raw.show(10, truncate=False)
RAW_COUNT = raw.count()
print("Total records (raw):", RAW_COUNT)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra |Saree                |Clothing |2       |6710      |20              |Credit Card     |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh    |T-Shirt              |Clothing |6       |680       |5               |Debit Card      |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao    |Hoodie               |Clothing |6       |3080      |12              |UPI             |P

Total records (raw): 1000


### 2.1 Duplicates

In [5]:
cols = raw.columns
exact_dup_rows = RAW_COUNT - raw.dropDuplicates().count()
print("Exact duplicate rows (extra copies):", exact_dup_rows)

print("\nRows involved in exact duplicates:")
raw.groupBy(cols).count().filter("count > 1").show(20, truncate=False)

# order_id repeated but rows differ -> NOT exact duplicates (reported only, kept)
same_id_diff_rows = (raw.dropDuplicates().groupBy("order_id").count().filter("count > 1").count())
print("order_ids that still repeat after removing exact duplicates (different content):", same_id_diff_rows)

Exact duplicate rows (extra copies): 10

Rows involved in exact duplicates:


+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|order_id|customer_id|customer_name   |product          |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|count|
+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|100329  |C0151      |Varun Verma     |Bedside Table    |Furniture      |9       |3530      |20              |Net Banking     |Vijayawada|Andhra Pradesh|2025-04-08|Delivered   |2    |
|100055  |C0270      |Pranav Iyer     |Dry Fruits Pack  |Grocery        |7       |930       |12              |Cash on Delivery|Coimbatore|Tamil Nadu    |2026-08-20|Delivered   |2    |
|100317  |C0264      |Siddharth Mishra|Kurta            |Clothing       |10     

order_ids that still repeat after removing exact duplicates (different content): 8


### 2.2 Null / blank values

In [6]:
blank = lambda c: F.col(c).isNull() | (F.trim(F.col(c)) == "")
raw.select([F.sum(blank(c).cast("int")).alias(c) for c in cols]).show(truncate=False)

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|0       |0          |10           |0      |0       |0       |0         |0               |8             |8   |0    |0         |0           |
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



### 2.3 Text-format inconsistencies (extra spaces, mixed capitalisation)

In [7]:
text_cols = ["customer_name","product","category","payment_method","city","state","order_status"]

# a) leading/trailing/multiple inner spaces
raw.select([F.sum((F.col(c) != F.trim(F.regexp_replace(F.col(c), r"\s+", " "))).cast("int")).alias(c)
            for c in text_cols]).show(truncate=False)

# b) same value written with different capitalisation / spacing
for c in ["category","city","payment_method","state","order_status"]:
    d = (raw.filter(F.col(c).isNotNull())
            .groupBy(F.lower(F.trim(F.col(c))).alias("key"))
            .agg(F.collect_set(F.col(c)).alias("variants"), F.count("*").alias("rows"))
            .filter(F.size("variants") > 1))
    print(f"--- {c}: values spelled in more than one way")
    d.show(truncate=False)

+-------------+-------+--------+--------------+----+-----+------------+
|customer_name|product|category|payment_method|city|state|order_status|
+-------------+-------+--------+--------------+----+-----+------------+
|3            |0      |0       |0             |3   |0    |0           |
+-------------+-------+--------+--------------+----+-----+------------+

--- category: values spelled in more than one way


+-----------+--------------------------+----+
|key        |variants                  |rows|
+-----------+--------------------------+----+
|beauty     |[beauty, Beauty]          |138 |
|electronics|[Electronics, ELECTRONICS]|104 |
|sports     |[Sports, sports, SPORTS]  |138 |
|clothing   |[Clothing, CLOTHING]      |109 |
+-----------+--------------------------+----+

--- city: values spelled in more than one way


+----------+-----------------------------------+----+
|key       |variants                           |rows|
+----------+-----------------------------------+----+
|delhi     |[delhi, Delhi, DELHI]              |84  |
|hyderabad |[HYDERABAD,  Hyderabad , Hyderabad]|97  |
|mumbai    |[ Mumbai , Mumbai]                 |107 |
|jaipur    |[Jaipur, JAIPUR]                   |85  |
|coimbatore|[coimbatore, Coimbatore]           |82  |
+----------+-----------------------------------+----+

--- payment_method: values spelled in more than one way


+---+--------+----+
|key|variants|rows|
+---+--------+----+
+---+--------+----+

--- state: values spelled in more than one way


+---+--------+----+
|key|variants|rows|
+---+--------+----+
+---+--------+----+

--- order_status: values spelled in more than one way


+---+--------+----+
|key|variants|rows|
+---+--------+----+
+---+--------+----+



### 2.4 Invalid numerical values

In [8]:
num = (raw.withColumn("q",  F.col("quantity").cast("double"))
          .withColumn("p",  F.col("unit_price").cast("double"))
          .withColumn("d",  F.col("discount_percent").cast("double")))

num.select(
    F.sum((F.col("q").isNull() & F.col("quantity").isNotNull()).cast("int")).alias("quantity_non_numeric"),
    F.sum((F.col("p").isNull() & F.col("unit_price").isNotNull()).cast("int")).alias("price_non_numeric"),
    F.sum((F.col("d").isNull() & F.col("discount_percent").isNotNull()).cast("int")).alias("discount_non_numeric"),
    F.sum((F.col("q") <= 0).cast("int")).alias("quantity_<=0"),
    F.sum((F.col("p") <= 0).cast("int")).alias("unit_price_<=0"),
    F.sum(((F.col("d") < 0) | (F.col("d") > 100)).cast("int")).alias("discount_outside_0_100"),
).show(truncate=False)

num.select(F.col("q").alias("quantity"), F.col("p").alias("unit_price"), F.col("d").alias("discount_percent")).summary("min","max","mean").show()

+--------------------+-----------------+--------------------+------------+--------------+----------------------+
|quantity_non_numeric|price_non_numeric|discount_non_numeric|quantity_<=0|unit_price_<=0|discount_outside_0_100|
+--------------------+-----------------+--------------------+------------+--------------+----------------------+
|0                   |0                |0                   |12          |8             |8                     |
+--------------------+-----------------+--------------------+------------+--------------+----------------------+



+-------+--------+----------+----------------+
|summary|quantity|unit_price|discount_percent|
+-------+--------+----------+----------------+
|    min|    -2.0|    -500.0|            -5.0|
|    max|    10.0|   81010.0|           120.0|
|   mean|   5.355|    5702.4|          19.948|
+-------+--------+----------+----------------+



### 2.5 Inconsistent / invalid dates

In [9]:
FORMATS = ["yyyy-MM-dd", "yyyy/MM/dd", "MM-dd-yyyy", "dd/MM/yyyy"]

d = raw.withColumn("is_iso", F.col("order_date").rlike(r"^\d{4}-\d{2}-\d{2}$"))
print("Dates NOT in the standard yyyy-MM-dd pattern:", d.filter(~F.col("is_iso")).count())
d.filter(~F.col("is_iso")).select("order_id","order_date").show(truncate=False)

# ISO-looking but impossible (e.g. 2026-02-30)?
iso_invalid = d.filter(F.col("is_iso") & F.to_date("order_date", "yyyy-MM-dd").isNull()).count()
print("ISO-pattern but impossible calendar dates:", iso_invalid)

Dates NOT in the standard yyyy-MM-dd pattern: 4


+--------+----------+
|order_id|order_date|
+--------+----------+
|100097  |31/02/2026|
|100358  |2025/07/14|
|100448  |not-a-date|
|100969  |09-15-2026|
+--------+----------+



ISO-pattern but impossible calendar dates: 2


## 3. Basic preprocessing → Silver layer

| # | Step | Method |
|---|------|--------|
| 1 | Exact duplicates | `dropDuplicates()` |
| 2 | Missing values | blank → NULL; `customer_name` recovered from other rows of the same `customer_id`, otherwise `"Unknown"`; `payment_method`/`city` → `"Unknown"` |
| 3 | Text cleaning | trim + collapse spaces on all text columns; `initcap` (Title Case) for customer name, category, city, payment method |
| 4 | Validation | `quantity <= 0` → NULL → median; `unit_price <= 0` → NULL → median price of the same product (global median fallback); `discount_percent` outside 0–100 → NULL → 0 (no discount) |
| 5 | Dates | parse several formats → one format `yyyy-MM-dd`; impossible / non-date values → NULL (record is kept) |
| 6 | Preserve | no rows removed except exact duplicates; unusual-but-valid values are left alone |

In [10]:
df = raw

# --- 1. remove exact duplicates -----------------------------------------------------------------
df = df.dropDuplicates()

# --- 3. text cleaning (done before null handling so blank strings become NULL) -----------------
for c in text_cols:
    df = df.withColumn(c, F.trim(F.regexp_replace(F.col(c), r"\s+", " ")))
    df = df.withColumn(c, F.when(F.col(c) == "", None).otherwise(F.col(c)))

for c in ["customer_name","category","payment_method","city"]:      # product names are left as-is (e.g. "T-Shirt")
    df = df.withColumn(c, F.initcap(F.col(c)))
# keep well-known acronyms/phrases tidy
df = df.withColumn("payment_method",
        F.when(F.upper("payment_method") == "UPI", "UPI")
         .when(F.lower("payment_method") == "cash on delivery", "Cash on Delivery")
         .otherwise(F.col("payment_method")))

# --- 2. missing values ---------------------------------------------------------------------------
# customer_name: reuse the most frequent name already known for that customer_id
name_map = (df.filter(F.col("customer_name").isNotNull())
              .groupBy("customer_id","customer_name").count()
              .withColumn("rn", F.row_number().over(Window.partitionBy("customer_id").orderBy(F.desc("count"))))
              .filter("rn = 1").select("customer_id", F.col("customer_name").alias("_name")))
df = (df.join(name_map, "customer_id", "left")
        .withColumn("customer_name", F.coalesce("customer_name", "_name", F.lit("Unknown")))
        .drop("_name"))
df = (df.withColumn("payment_method", F.coalesce("payment_method", F.lit("Unknown")))
        .withColumn("city",           F.coalesce("city",           F.lit("Unknown"))))

# --- 4. numeric validation -----------------------------------------------------------------------
df = (df.withColumn("quantity",         F.col("quantity").cast("int"))
        .withColumn("unit_price",       F.col("unit_price").cast("double"))
        .withColumn("discount_percent", F.col("discount_percent").cast("double")))

df = (df.withColumn("quantity",         F.when(F.col("quantity") > 0, F.col("quantity")))
        .withColumn("unit_price",       F.when(F.col("unit_price") > 0, F.col("unit_price")))
        .withColumn("discount_percent", F.when(F.col("discount_percent").between(0, 100), F.col("discount_percent"))))

med_qty   = df.approxQuantile("quantity",   [0.5], 0.001)[0]
med_price = df.approxQuantile("unit_price", [0.5], 0.001)[0]
prod_med  = (df.groupBy("product")
               .agg(F.expr("percentile_approx(unit_price, 0.5)").alias("_pmed")))

df = (df.join(prod_med, "product", "left")
        .withColumn("quantity",         F.coalesce("quantity", F.lit(int(med_qty))))
        .withColumn("unit_price",       F.coalesce("unit_price", "_pmed", F.lit(med_price)))
        .withColumn("discount_percent", F.coalesce("discount_percent", F.lit(0.0)))
        .drop("_pmed"))
print(f"Imputation values -> median quantity: {med_qty}, global median unit_price: {med_price}")

# --- 5. dates -> one format (yyyy-MM-dd); invalid -> NULL ------------------------------------------
parsed = F.coalesce(*[F.to_date(F.col("order_date"), f) for f in FORMATS])
df = df.withColumn("order_date", F.date_format(parsed, "yyyy-MM-dd"))

silver = df.select(cols).orderBy("order_id")
silver.cache()
SILVER_COUNT = silver.count()
print("Silver rows:", SILVER_COUNT)

Imputation values -> median quantity: 5.0, global median unit_price: 1840.0



[Stage 115:=====>                                                (22 + 1) / 200]




[Stage 115:==========>                                           (39 + 1) / 200]




[Stage 115:=================================>                   (125 + 1) / 200]




[Stage 115:=======================================>             (150 + 1) / 200]




[Stage 119:=========>                                            (34 + 1) / 200]




[Stage 119:==================>                                   (67 + 1) / 200]




[Stage 119:==========================>                           (98 + 1) / 200]




[Stage 119:=====================================>               (143 + 1) / 200]




[Stage 119:=============================================>       (170 + 1) / 200]



Silver rows: 990


### Save to the Silver layer

In [11]:
# Spark write -> data_lake/silver/preprocessed_sales/ (CSV part file(s) + _SUCCESS)
(silver.coalesce(1).write.mode("overwrite").option("header", True)
       .csv(SILVER))

# Also export a single, clearly named file for the GitHub submission: silver/sales_silver.csv
os.makedirs(f"{BASE}/silver", exist_ok=True)
part = glob.glob(f"{SILVER}/part-*.csv")[0]
shutil.copyfile(part, f"{BASE}/silver/sales_silver.csv")

for root, dirs, fs in os.walk(f"{BASE}/silver"):
    print(root + "/", fs)

data_lake/silver/ ['sales_silver.csv']
data_lake/silver/preprocessed_sales/ ['._SUCCESS.crc', '_SUCCESS', '.part-00000-8f4c8ddf-2bee-47fb-bad5-b6a3a049cd0c-c000.csv.crc', 'part-00000-8f4c8ddf-2bee-47fb-bad5-b6a3a049cd0c-c000.csv']


## 4. Verification

In [12]:
print("Rows BEFORE preprocessing (Bronze):", RAW_COUNT)
print("Rows AFTER  preprocessing (Silver):", SILVER_COUNT)
print("Rows removed (exact duplicates)   :", RAW_COUNT - SILVER_COUNT)

# Read the Silver data back with Spark
silver_back = spark.read.option("header", True).option("inferSchema", True).csv(SILVER)
print("Rows read back from Silver:", silver_back.count())
silver_back.printSchema()
silver_back.show(10, truncate=False)

Rows BEFORE preprocessing (Bronze): 1000
Rows AFTER  preprocessing (Silver): 990
Rows removed (exact duplicates)   : 10


Rows read back from Silver: 990
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+

In [13]:
# Re-run the quality checks on Silver (computed by Spark)
s = silver_back
checks = s.select(
    F.sum((F.col("quantity") <= 0).cast("int")).alias("quantity_<=0"),
    F.sum((F.col("unit_price") <= 0).cast("int")).alias("unit_price_<=0"),
    F.sum(((F.col("discount_percent") < 0) | (F.col("discount_percent") > 100)).cast("int")).alias("discount_invalid"),
    F.sum(F.col("order_date").isNull().cast("int")).alias("null_dates_(invalid_in_raw)"),
    F.sum((~F.col("order_date").rlike(r"^\d{4}-\d{2}-\d{2}$") & F.col("order_date").isNotNull()).cast("int")).alias("non_iso_dates"),
)
checks.show(truncate=False)

print("Remaining exact duplicates:", s.count() - s.dropDuplicates().count())
print("Nulls per column:")
s.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in s.columns]).show(truncate=False)
for c in ["category","city","payment_method"]:
    print(f"Distinct {c}:"); s.groupBy(c).count().orderBy(c).show(30, truncate=False)

+------------+--------------+----------------+---------------------------+-------------+
|quantity_<=0|unit_price_<=0|discount_invalid|null_dates_(invalid_in_raw)|non_iso_dates|
+------------+--------------+----------------+---------------------------+-------------+
|0           |0             |0               |4                          |0            |
+------------+--------------+----------------+---------------------------+-------------+



Remaining exact duplicates: 0
Nulls per column:


+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|0       |0          |0            |0      |0       |0       |0         |0               |0             |0   |0    |4         |0           |
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+

Distinct category:


+---------------+-----+
|category       |count|
+---------------+-----+
|Beauty         |137  |
|Books          |127  |
|Clothing       |108  |
|Electronics    |104  |
|Furniture      |129  |
|Grocery        |134  |
|Home Appliances|113  |
|Sports         |138  |
+---------------+-----+

Distinct city:
+----------+-----+
|city      |count|
+----------+-----+
|Bengaluru |90   |
|Chennai   |88   |
|Coimbatore|81   |
|Delhi     |83   |
|Hyderabad |97   |
|Jaipur    |83   |
|Kolkata   |81   |
|Lucknow   |89   |
|Mumbai    |105  |
|Pune      |81   |
|Unknown   |8    |
|Vijayawada|104  |
+----------+-----+

Distinct payment_method:


+----------------+-----+
|payment_method  |count|
+----------------+-----+
|Cash on Delivery|206  |
|Credit Card     |203  |
|Debit Card      |199  |
|Net Banking     |204  |
|UPI             |170  |
|Unknown         |8    |
+----------------+-----+



### Preprocessing operations performed
The summary below is also printed with the real counts by the next cell.

In [14]:
raw_blank = {c: raw.filter(F.col(c).isNull() | (F.trim(F.col(c)) == "")).count() for c in ["customer_name","payment_method","city"]}
bad_q = num.filter(F.col("q") <= 0).count()
bad_p = num.filter(F.col("p") <= 0).count()
bad_d = num.filter((F.col("d") < 0) | (F.col("d") > 100)).count()
dparsed = raw.withColumn("p", F.coalesce(*[F.to_date("order_date", f) for f in FORMATS]))
bad_dates = dparsed.filter("p IS NULL").count()
fixed_fmt = dparsed.filter(F.col("p").isNotNull() & ~F.col("order_date").rlike(r"^\d{4}-\d{2}-\d{2}$")).count()

print(f"""
SUMMARY
1. Removed {RAW_COUNT - SILVER_COUNT} exact duplicate rows            ({RAW_COUNT} -> {SILVER_COUNT} rows)
2. Missing values handled: {raw_blank}
   (customer_name recovered via customer_id where possible, else 'Unknown')
3. Trimmed spaces and standardised capitalisation (customer name, category, city, payment method)
4. Invalid numerics replaced: quantity<=0: {bad_q} (median), unit_price<=0: {bad_p} (product median),
   discount outside 0-100: {bad_d} (set to 0)  -- rows kept
5. Dates converted to yyyy-MM-dd: {fixed_fmt} re-formatted, {bad_dates} impossible/non-date values set to NULL (rows kept)
6. No other rows removed; unusual-but-valid values were left unchanged
""")


SUMMARY
1. Removed 10 exact duplicate rows            (1000 -> 990 rows)
2. Missing values handled: {'customer_name': 10, 'payment_method': 8, 'city': 8}
   (customer_name recovered via customer_id where possible, else 'Unknown')
3. Trimmed spaces and standardised capitalisation (customer name, category, city, payment method)
4. Invalid numerics replaced: quantity<=0: 12 (median), unit_price<=0: 8 (product median),
   discount outside 0-100: 8 (set to 0)  -- rows kept
5. Dates converted to yyyy-MM-dd: 2 re-formatted, 4 impossible/non-date values set to NULL (rows kept)
6. No other rows removed; unusual-but-valid values were left unchanged



In [15]:
spark.stop()